# Programming for Data Science — Assignment 06
## Assignment 06: Predicting College Dropout with Logistic Regression
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Assignments/Assignment_06_Logistic_Regression.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

This assignment goes with the in-class notebook **`10_Functions_Lambdas_Comprehensions.ipynb`**. You fit a logistic regression that predicts which college students drop out, check how well it predicts students it has never seen, and then build a model of your own. Next week you will fit a neural network to the same data and compare the two.

About this notebook:  
The design by Chenguang Pan, in collaboration with AI. I am responsible for its accuracy.

---
### 📋 How this assignment works (Read me carefully)

- **Run on Google Colab.** Open this notebook with the badge above, then *File ▸ Save a copy in Drive* so you have your own copy to edit.
- **The data.** Records of 4,424 students who started undergraduate programs at a college in Portugal, shared through the UCI Machine Learning Repository. For each student you know the program, a few facts from enrollment, how the first semester went, and the outcome: where the student stood at the end of the program's normal length (dropped out, still enrolled, or graduated).
- **Two files.** The students are split at random into a **training file** and a **test file**, with 2,212 students each. You fit every model on the training file and check it on the test file. A model usually looks a little better on the students it was fit to, so the test file is the honest check: it shows how the model does on students it has never seen, which is what a college would face with next year's students.
- **The question.** At the end of each student's first semester, the college wants to know who is likely to drop out, so that advisors can reach out before the second semester starts. Your models answer 1 ("likely to drop out") or 0 for every student.
- **Three parts.** Part 1: four small functions that you use later. Part 2: load the data, then run a worked example that fits a logistic regression and checks it on the test file. Part 3: fit a logistic regression of your own, check it on the test file, and explain it.
- **Tools.** What we covered in Session 10 (your own functions, default arguments, `lambda`, comprehensions, and `.apply()`), plus `statsmodels`, which you met in Assignment 05.
- **AI help.** Try each part on your own first. After that you may ask an AI tool to explain an error, a concept, or a regression output, as the syllabus describes. Name the tool and what you used it for in `ai_note`, in the cell under Before you start. Before you ask an AI tool, download your notebook and keep that copy as your draft; the Submission section says what to upload.
- **Handle with care.** These are real, anonymized students. A higher dropout rate for a group in this file describes an average and says nothing certain about any one student. When a model flags someone, it means an advisor should check in.
- Use **clear variable names**, give every function a **docstring**, and drop a **comment** on the tricky lines.

**Total: 100 points** — Part 1: 20 · Part 2: 20 · Part 3: 50 · Code quality: 10.

---
### ✍️ Before you start
Put your name in the next cell, and write down any AI tool you used (leave it empty if you used none). Then run the setup cell right after it.

In [ ]:
student_name = ""   # <-- type your name between the quotes
ai_note = ""        # <-- e.g. "ChatGPT: explained the odds ratios in Part 2" (leave "" if none)
print("Submission by:", student_name)

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf   # logistic regression with formulas, from Part 2 on

# The student records live in the public course repo, so they load straight from the web.
TRAIN_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/dropout_train.csv"   # fit models here
TEST_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/dropout_test.csv"     # check them here

print("pandas", pd.__version__, "| NumPy", np.__version__)
print("Setup complete ✅")

#### 🗂️ What is in the data

Both files have one row per student and the same 15 columns.

| Column (as written in the file) | What it holds |
|---|---|
| `student_id` | an anonymous ID, added for this assignment |
| `Course` | the undergraduate program, one of 17 |
| `Daytime/evening attendance` | `daytime` or `evening` classes |
| `Age at enrollment` | age in years when the student enrolled |
| `Gender` | `female` or `male`, as recorded by the college |
| `Scholarship holder` | 1 = has a scholarship, 0 = does not |
| `Debtor` | 1 = owes the college money, 0 = does not |
| `Tuition fees up to date` | 1 = tuition is paid up to date, 0 = it is not |
| `Displaced` | 1 = yes, 0 = no. The source gives no definition; it most likely marks students who moved away from home to study |
| `Admission grade` | the grade the student was admitted with, on a scale from 0 to 200 |
| `Curricular units 1st sem (enrolled)` | how many courses ("curricular units") the student signed up for in the first semester |
| `Curricular units 1st sem (evaluations)` | how many evaluations (exams and other graded work) the student took in those courses |
| `Curricular units 1st sem (approved)` | how many of those courses the student passed |
| `Curricular units 1st sem (grade)` | the average first-semester grade, on a scale from 0 to 20 (0 when the student passed no course) |
| `Target` | the outcome: `Dropout`, `Enrolled` (still enrolled at the end of the program's normal length), or `Graduate` |

The column names are exactly as the source wrote them, with spaces, parentheses, a slash, and one invisible tab at the end of `Daytime/evening attendance`, and Function 1 below turns them into clean names. The model formulas in Parts 2 and 3 need clean names.

---
## Part 1 — Warm-up: four small functions (20 points)

Each function below has an exact specification and a few example calls with the results they should give. Write each function in its cell and run the examples to check it. Wrap each example in `print()`: a cell shows only the value of its last line, and nothing at all when that value is `None`. You will use them in Parts 2 and 3.

#### Function 1 — `snake_case(name)`
**Difficulty:** ★★☆☆☆  **Points:** 5  
**Skills:** `def`, docstring, a comprehension over the characters of a string, `.isalnum()`, `.split()`, `.join()`

Return `name` in lowercase, with every run of characters that are not letters or digits turned into a single underscore, and with no underscore at the start or the end.

```python
snake_case("Age at enrollment")                  # 'age_at_enrollment'
snake_case("Curricular units 1st sem (grade)")   # 'curricular_units_1st_sem_grade'
snake_case("Daytime/evening attendance\t")       # 'daytime_evening_attendance'
```

🧰 `ch.isalnum()` is `True` when the character `ch` is a letter or a digit. With no argument, `.split()` cuts a string at every run of whitespace and drops the empty pieces, so `"  a  b ".split()` gives `['a', 'b']`. And `"_".join(["a", "b"])` glues a list of strings into one with `_` between them, giving `'a_b'`; `"".join(...)` glues them with nothing in between.

In [ ]:
# Your code here

#### Function 2 — `accuracy(truth, guesses)`
**Difficulty:** ★★☆☆☆  **Points:** 5  
**Skills:** `def`, `zip`, a list comprehension, `sum`, `len`

Return the share of positions where `truth` and `guesses` hold the same value, as a number between 0 and 1. Pair the two up with `zip` inside a comprehension. Both arguments are lists, or pandas Series, of the same length.

```python
accuracy([1, 0, 1, 1], [1, 0, 0, 1])   # 0.75
accuracy([0, 0, 0], [1, 1, 1])         # 0.0
```

In [ ]:
# Your code here

#### Function 3 — `share_passed(approved, enrolled, if_none=None)`
**Difficulty:** ★★☆☆☆  **Points:** 5  
**Skills:** `def`, a default argument, `if`, an early `return`

Return `approved / enrolled`, the share of first-semester courses a student passed. When `enrolled` is 0 there is nothing to divide by, so return `if_none` instead. Its default value is `None`.

```python
share_passed(4, 6)                # 0.6666666666666666
share_passed(0, 0)                # None
share_passed(0, 0, if_none=1.0)   # 1.0
```

Some students in the training file enrolled in no courses in their first semester, so the zero case does come up.

In [ ]:
# Your code here

#### Function 4 — `rate_by(df, column, outcome="dropout", ndigits=3)`
**Difficulty:** ★★★☆☆  **Points:** 5  
**Skills:** `def`, default arguments, `groupby`, a dict comprehension, `sorted(key=lambda ...)`

Return a dictionary that maps each value of `column` to the mean of the `outcome` column among the rows with that value, rounded to `ndigits`. When the `outcome` column holds 1 for "dropped out" and 0 otherwise, that mean is the dropout rate. Build the dictionary with a dict comprehension. The order of the keys does not matter, and if a number prints as `np.float64(0.5)`, wrap it in `float()`.

Then print the pairs in `rate_by(toy, "course")` from the highest rate to the lowest, using `sorted()` with a `lambda` as the key.

```python
rate_by(toy, "course")   # {'Agronomy': 1.0, 'Nursing': 0.0, 'Tourism': 0.5}
# highest to lowest:     [('Agronomy', 1.0), ('Tourism', 0.5), ('Nursing', 0.0)]
```

In [ ]:
toy = pd.DataFrame({"course":  ["Nursing", "Tourism", "Nursing", "Tourism", "Agronomy"],
                    "dropout": [0, 1, 0, 0, 1]})

# Your code here

---
## Part 2 — Load the data and fit a first model (20 points)

### 2.1 Load and prepare (10 points)
**Skills:** `pd.read_csv`, a dict comprehension, `.rename(columns=...)`, `.apply()` with a `lambda`

- Load the training file into a DataFrame named `train` and the test file into one named `test`. The URLs are in the setup cell.
- Rename the columns of both tables with your `snake_case` function and a dict comprehension.
- Add a column `dropout` to both tables that holds the number 1 when `target` is `"Dropout"` and 0 otherwise (numbers, not `True` and `False`). Build it with `.apply()` and a `lambda`.

Then run Checkpoint 1.

In [ ]:
# Your code here

In [ ]:
# 🔎 Checkpoint 1 — run me, do not edit
import pandas as pd

expected = ["student_id", "course", "daytime_evening_attendance", "age_at_enrollment", "gender",
            "scholarship_holder", "debtor", "tuition_fees_up_to_date", "displaced", "admission_grade",
            "curricular_units_1st_sem_enrolled", "curricular_units_1st_sem_evaluations",
            "curricular_units_1st_sem_approved", "curricular_units_1st_sem_grade", "target"]

def dropout_ok(df):
    """Return True when df has a numeric dropout column that is 1 exactly when target is 'Dropout'."""
    return ("dropout" in df.columns and "target" in df.columns
            and df["dropout"].dtype.kind in "iuf"             # numbers, not True/False
            and bool((df["dropout"] == (df["target"] == "Dropout").astype(int)).all()))

tr, te = globals().get("train"), globals().get("test")
if not (isinstance(tr, pd.DataFrame) and isinstance(te, pd.DataFrame)):
    print("❌ load the two files into DataFrames named train and test first")
else:
    checks = {
        "train and test have 2,212 students each": len(tr) == 2212 and len(te) == 2212,
        "both tables start with the 15 expected column names":
            list(tr.columns[:15]) == expected and list(te.columns[:15]) == expected,
        "both tables have a dropout column of 0s and 1s (numbers, not True/False) that is 1 exactly when target is 'Dropout'":
            dropout_ok(tr) and dropout_ok(te),
    }
    for name, ok in checks.items():
        print("✅" if ok else "❌", name)
    for got, want in zip(te.columns, expected):
        if got != want:
            print(f"   the first name in test that differs: {got!r}, expected {want!r}")
            break

### 2.2 Worked example: a first logistic regression (10 points)  
  
This 2.2 section is a worked example for you to follow. Please read it carefully and learn how to fit a logistic regression with `statsmodels`, check it on the test file, and interpret the output. You will fit your own logistic regression model in Part 3.

**What the model does.** Logistic regression predicts the probability $p$ that a student drops out (`dropout` = 1). It works with the **odds** of dropping out, $p/(1-p)$: a probability of 0.2 gives odds of 0.2/0.8 = 0.25, one student who drops out for every four who stay. The model writes the log of the odds as a weighted sum of the predictors,

$$\log\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_k x_k,$$

and turning the log-odds back into a probability always gives a number between 0 and 1. A positive coefficient means that higher values of that predictor go with a higher chance of dropping out, and a negative one goes with a lower chance.

**The formula.** `statsmodels` lets you write the model as a string: the outcome, a `~`, then the predictors joined by `+`. The example uses three predictors: whether tuition is up to date, whether the student has a scholarship, and age at enrollment. Run the three cells below in order, read the outputs, and then answer the questions under them. The third cell uses your `accuracy` function from Part 1.

In [ ]:
# Step 1: fit the model on the training students and print the results — run me, do not edit
formula = "dropout ~ tuition_fees_up_to_date + scholarship_holder + age_at_enrollment"
demo_model = smf.logit(formula, data=train).fit()
print(demo_model.summary())

#### 🔍 How to read the summary

- **Top block.** `No. Observations` is the number of training students used, and `converged` should say `True`. `Pseudo R-squ.` is a rough measure of fit: 0 means the predictors do no better than the overall dropout rate, and it rises whenever you add a predictor, so use the test file to compare models (logistic regression has no exact R² like the one in Assignment 05). `LLR p-value` tests whether the predictors, taken together, help at all.
- **Middle block, one row per coefficient.** `coef` is the estimate on the log-odds scale, `std err` its standard error, `z` the estimate divided by its standard error, and `P>|z|` the p value for the test that the coefficient is 0. The last two columns give the 95% confidence interval. A p value printed as `0.000` is below 0.0005, so report it as p < 0.001, and a number such as `4.064e-141` means 4.064 × 10⁻¹⁴¹.
- **Odds ratios.** A log-odds coefficient is hard to read directly, so we turn it into an odds ratio, $e^{\text{coef}}$. It is the factor by which the odds of dropping out change when that predictor goes up by 1 and the other predictors stay the same: below 1 means lower odds, above 1 higher odds. An odds ratio of 0.28 means 72% lower odds; the change in the probability itself is smaller and depends on where a student starts. Skip the `Intercept` row: it is the odds for a student whose predictors are all 0, which describes no real student here.

In [ ]:
# Step 2: the odds ratios, one per coefficient — run me, do not edit
odds_ratios = np.exp(demo_model.params)
print(odds_ratios.round(3))

In [ ]:
# Step 3: check the model on the test students — run me, do not edit
test_prob = demo_model.predict(test)            # each test student's probability of dropping out
test_guess = (test_prob >= 0.5).astype(int)     # 1 when that probability is at least one half

print(test_prob.head().round(3), "\n")
print("test accuracy:   ", round(accuracy(test["dropout"], test_guess), 3))   # your Function 2
nobody = [0 for _ in range(len(test))]          # the guess "stays" (0) for every student
print("nobody drops out:", round(accuracy(test["dropout"], nobody), 3))
print()
# rows: the actual outcome; columns: the model's guess
print(pd.crosstab(test["dropout"], test_guess, rownames=["actual"], colnames=["predicted"]))

**Your turn (10 points).** In the cell below, answer each question in a sentence or two.
1. Which of the three predictors go with a higher chance of dropping out, and which with a lower chance? How can you tell from the summary?
2. What is the odds ratio for `scholarship_holder`? Say in plain words what it means for the odds of dropping out.
3. How does the model's test accuracy compare with "nobody drops out"? Of the test students who dropped out, how many did the model catch? (The bottom row of the table counts the students who dropped out.)

*Double-click this cell and write your answers here.*

---
## Part 3 — Your own model (50 points)

Now you fit a logistic regression of your own: choose the predictors, fit the model on the training students, check it on the test students, and explain it. The worked example leaves out most of the columns, including everything about the first semester, so look over the data table at the top before you choose. Your reason for a predictor can come from the data, such as a dropout rate from your `rate_by` function, or from what you know about students; either is fine.

🧰 **Toolbox**
- **Formulas.** Add predictors with `+`, as in `"dropout ~ a + b + c"`. The 0/1 columns go in as they are. A text column such as `gender` can go in as it is too: statsmodels codes it as 0/1 for you, with the value that comes first in the alphabet as the comparison group, so its row in the summary reads `gender[T.male]` (male students compared with female students).
- **Never use the answer.** `target` and `dropout` are the outcome, so leave them out of the formula: with either one, the model reads the answer and scores about 100%. `student_id` is an ID and tells the model nothing.
- **Dropouts caught.** The share of dropouts a model catches is its accuracy among the students who dropped out: `is_dropout = test["dropout"] == 1`, then `accuracy(test["dropout"][is_dropout], guesses[is_dropout])`.
- **When a fit complains.** An error that ends in `NameError: name 'x' is not defined` means `x` is not a column of that table: check the spelling against `train.columns`, and if you built `x` yourself, build it in `test` too. A `ConvergenceWarning` or a warning about perfect separation means a predictor pins the outcome down, so drop it. If `No. Observations` in the summary is below 2,212, a column has missing values. Always give `predict` a table, as in `model.predict(test)`.
- **A predictor you build (optional).** You may add a predictor that is not in the file, such as the share of first-semester courses passed. Build it in both `train` and `test` under the same name, with a number for every student: statsmodels drops rows with a missing value when it fits and returns `NaN` for them when it predicts, so pass `if_none=` a number to `share_passed`. When a rule needs more than one column, write a small function that takes one row and call `.apply()` on the whole DataFrame with `axis=1`. It runs the function once for each row and hands it that row, so inside it you read any value with `row["column_name"]`:

```python
def share_of_row(row):
    """Return the share of first-semester courses passed (0.5 for students with no courses)."""
    return share_passed(row["curricular_units_1st_sem_approved"],
                        row["curricular_units_1st_sem_enrolled"], if_none=0.5)

train["share"] = train.apply(share_of_row, axis=1)   # one number per student; do the same for test
```

### 3.1 Check and prepare the data (10 points)

Before you fit your model, get both tables ready.
- Check each column of `train` and `test` for missing values; `.isna().sum()` counts them column by column. If only a few rows have missing values, drop those rows with `.dropna()`, and print how many you dropped. Skip this if no missing values exist.
- Drop the `course` column from both tables. We do not need it for the model.

In [ ]:
# Your code here

### 3.2 Build your logistic regression model (25 points)

- Choose your predictors and write them in a formula. Use a set of predictors that differs from the worked example's.
- Fit the model on `train`, then print its summary and its odds ratios.
- Predict both the training and the test students, and turn the probabilities into 0/1 guesses with the 0.5 cutoff, as in Step 3 of the worked example. Print the accuracy on each file, the share of the test students who dropped out that your model catches, and the table of actual against predicted outcomes for the test students.
- Store these numbers in a dictionary named `final_results` with the keys `"formula"`, `"train_accuracy"`, `"test_accuracy"`, and `"dropouts_caught"`. You will compare them with a neural network next week. Then run Checkpoint 2.

In [ ]:
# Your code here: choose your predictors, fit the model, and print its summary and odds ratios

In [ ]:
# Your code here: predict both files, score the model, and store final_results

In [ ]:
# 🔎 Checkpoint 2 — run me, do not edit
import re

results = globals().get("final_results")
if not isinstance(results, dict):
    print("❌ store your model's numbers in a dictionary named final_results")
else:
    if re.search(r"\b(target|dropout)\b", str(results.get("formula", "")).split("~")[-1]):
        print("❌ your formula uses target or dropout as a predictor; leave the answer out")
    for key in ["formula", "train_accuracy", "test_accuracy", "dropouts_caught"]:
        value = results.get(key)
        if key not in results:
            print(f"❌ final_results has no key {key!r}")
        elif key == "formula" and not (isinstance(value, str) and "~" in value):
            print("❌ final_results['formula'] should be the formula string of your model")
        elif key != "formula" and not (isinstance(value, (int, float)) and 0 <= value <= 1):
            print(f"❌ final_results[{key!r}] should be a number between 0 and 1")
        else:
            print(f"✅ {key}: {value}")

### 3.3 Explain your model (15 points)

First, in the code cell, look at the test students who dropped out but whose guess from your model is 0. With your model's 0/1 guesses for the test students in `guesses`, `test[(test["dropout"] == 1) & (guesses == 0)]` picks them out. Then, in the text cell below, answer in 6 to 10 sentences:
1. Why did you choose these predictors? Your reason can come from the data or from what you know about students.
2. Pick two predictors in your model and say what their odds ratios mean. You may ask an AI tool to help you read the output; if you do, check one of its numbers against your own summary and say that you did.
3. How does your model do on the test students, compared with the worked example and with "nobody drops out"? How many test dropouts does it miss, and what do they have in common?
4. Would you use gender or age in a model that a real college runs on its students? Why or why not?

In [ ]:
# Your code here: the test dropouts your model misses

*Double-click this cell and write your answers here.*

---
## 📤 Submission

1. In Colab, run *Runtime ▸ Run all* and make sure every cell finishes **without errors**, with ✅ in both checkpoints.
2. Download your notebook (*File ▸ Download ▸ Download .ipynb*) and upload it to Canvas by **Monday, Nov 16, 11:59 pm**.
3. If you used AI, also upload the draft you saved before you used it, as the syllabus asks.

**Code quality (10 points)** covers the whole notebook: a docstring on every function, clear names, comments on the tricky lines, and your Part 1 functions reused where they fit instead of copied code.

---
**Data credit.** Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict Students' Dropout and Academic Success* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5MC89. Shared under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes for this assignment: 13 of the 36 student columns kept, plus `Target`; program names, `daytime`/`evening`, and `female`/`male` written in place of number codes; first-semester grades rounded to 2 decimals; a `student_id` added; the students split at random into two halves with nearly the same share of dropouts; values separated by commas instead of semicolons.